In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from copy import deepcopy

# ══════════════════════════════════════════════
#  1. DROPOUT FROM SCRATCH
# ══════════════════════════════════════════════

class DropoutScratch(nn.Module):
    """
    Dropout — randomly zeroes neurons during TRAINING.
    At INFERENCE, all neurons are active (no randomness).

    Training:   mask = Bernoulli(1 - p)
                out  = x * mask / (1 - p)     ← "inverted dropout" scaling
    Inference:  out  = x                       ← no change needed
    """

    def __init__(self, p: float = 0.5):
        super().__init__()
        self.p = p    # probability of DROPPING a neuron

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not self.training or self.p == 0:
            return x    # inference mode — pass through unchanged

        # Generate binary mask: 1 = keep, 0 = drop
        mask = (torch.rand_like(x) > self.p).float()

        # Scale up survivors so expected value stays the same
        return x * mask / (1 - self.p)


# ══════════════════════════════════════════════
#  2. L2 REGULARIZATION (WEIGHT DECAY) FROM SCRATCH
# ══════════════════════════════════════════════

def l2_penalty(model: nn.Module) -> torch.Tensor:
    """
    Compute L2 penalty = sum of squared weights.

        L2 = Σ ||w||²

    Added to the loss:  total_loss = data_loss + λ * L2
    Effect: pushes weights toward zero → prevents overfitting.
    """
    penalty = torch.tensor(0.0)
    for name, param in model.named_parameters():
        if "weight" in name:    # regularize weights, not biases
            penalty = penalty + (param ** 2).sum()
    return penalty


class SGDWithWeightDecay:
    """
    SGD with explicit weight decay — equivalent to L2 regularization.

        θ = θ - lr * (∇L + λ * θ)
        θ = (1 - lr*λ) * θ - lr * ∇L     ← "decay" the weights each step
    """

    def __init__(self, params, lr: float = 0.01, weight_decay: float = 0.0):
        self.params = list(params)
        self.lr = lr
        self.wd = weight_decay

    def step(self):
        with torch.no_grad():
            for p in self.params:
                if p.grad is not None:
                    p -= self.lr * (p.grad + self.wd * p)

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()


# ══════════════════════════════════════════════
#  3. EARLY STOPPING FROM SCRATCH
# ══════════════════════════════════════════════

class EarlyStopping:
    """
    Stop training when validation loss stops improving.

    Tracks the best val loss and saves the best model weights.
    If val loss hasn't improved for `patience` epochs → stop.
    """

    def __init__(self, patience: int = 10, min_delta: float = 1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_loss  = float("inf")
        self.counter    = 0
        self.best_model = None
        self.stopped_epoch = 0

    def __call__(self, val_loss: float, model: nn.Module, epoch: int) -> bool:
        if val_loss < self.best_loss - self.min_delta:
            # Improvement — reset counter and save model
            self.best_loss  = val_loss
            self.counter    = 0
            self.best_model = deepcopy(model.state_dict())
            self.stopped_epoch = epoch
            return False    # keep training
        else:
            # No improvement
            self.counter += 1
            if self.counter >= self.patience:
                return True     # STOP
            return False        # keep going

    def restore_best(self, model: nn.Module):
        """Load the best weights back into the model."""
        if self.best_model is not None:
            model.load_state_dict(self.best_model)


# ══════════════════════════════════════════════
#  4. MODELS FOR COMPARISON
# ══════════════════════════════════════════════

class OverfitNet(nn.Module):
    """Deliberately too big — will overfit."""
    def __init__(self, n_in: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, 64),  nn.ReLU(),
            nn.Linear(64, 1),    nn.Sigmoid(),
        )
    def forward(self, x):
        return self.net(x)


class UnderfitNet(nn.Module):
    """Deliberately too small — will underfit."""
    def __init__(self, n_in: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, 2), nn.ReLU(),
            nn.Linear(2, 1),    nn.Sigmoid(),
        )
    def forward(self, x):
        return self.net(x)


class RegularizedNet(nn.Module):
    """Right-sized model with dropout — balanced."""
    def __init__(self, n_in: int, dropout: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, 128),  nn.ReLU(),
            DropoutScratch(dropout),
            nn.Linear(128, 128),   nn.ReLU(),
            DropoutScratch(dropout),
            nn.Linear(128, 64),    nn.ReLU(),
            DropoutScratch(dropout),
            nn.Linear(64, 1),      nn.Sigmoid(),
        )
    def forward(self, x):
        return self.net(x)


# ══════════════════════════════════════════════
#  5. TRAINING + EVALUATION LOOP
# ══════════════════════════════════════════════

def train_model(model, X_train, y_train, X_val, y_val,
                epochs: int = 200, lr: float = 0.01,
                weight_decay: float = 0.0,
                early_stopping: EarlyStopping = None,
                label: str = "") -> dict:
    """Train and record train/val loss per epoch."""

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCELoss()

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(1, epochs + 1):
        # ── Train ──
        model.train()
        pred = model(X_train)
        loss = criterion(pred, y_train)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss = loss.item()
        train_acc  = ((pred >= 0.5).float() == y_train).float().mean().item()

        # ── Validate ──
        model.eval()
        with torch.no_grad():
            val_pred = model(X_val)
            val_loss = criterion(val_pred, y_val).item()
            val_acc  = ((val_pred >= 0.5).float() == y_val).float().mean().item()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        # ── Early stopping check ──
        if early_stopping is not None:
            if early_stopping(val_loss, model, epoch):
                early_stopping.restore_best(model)
                history["stopped_at"] = epoch
                break

    if "stopped_at" not in history:
        history["stopped_at"] = epochs

    return history


# ══════════════════════════════════════════════
#  6. PRINT LOSS CURVES (ASCII)
# ══════════════════════════════════════════════

def print_curves(history: dict, label: str, n_points: int = 10):
    """Print loss curves as ASCII for train vs val."""
    tl = history["train_loss"]
    vl = history["val_loss"]
    total = len(tl)
    step = max(1, total // n_points)
    stopped = history.get("stopped_at", total)

    print(f"  {'Epoch':<8} {'Train Loss':>12} {'Val Loss':>12} {'Gap':>10} {'Signal'}")
    print(f"  {'─'*8} {'─'*12} {'─'*12} {'─'*10} {'─'*18}")

    for i in range(0, total, step):
        gap = vl[i] - tl[i]
        if gap > 0.15:
            signal = "⚠ OVERFITTING"
        elif tl[i] > 0.5 and vl[i] > 0.5:
            signal = "⚠ UNDERFITTING"
        else:
            signal = "✓ good"
        print(f"  {i+1:<8} {tl[i]:>12.4f} {vl[i]:>12.4f} {gap:>+10.4f} {signal}")

    # Final
    i = total - 1
    gap = vl[i] - tl[i]
    if i % step != 0:
        signal = "⚠ OVERFITTING" if gap > 0.15 else "⚠ UNDERFITTING" if tl[i] > 0.5 else "✓ good"
        print(f"  {i+1:<8} {tl[i]:>12.4f} {vl[i]:>12.4f} {gap:>+10.4f} {signal}")

    if stopped < total:
        print(f"  ⏹ Early stopped at epoch {stopped}")

    # Visual bar comparison
    print(f"\n  Train loss: ", end="")
    bar_t = int(30 * min(tl[-1], 1.0))
    print("█" * bar_t + "░" * (30 - bar_t) + f" {tl[-1]:.4f}")
    print(f"  Val   loss: ", end="")
    bar_v = int(30 * min(vl[-1], 1.0))
    print("█" * bar_v + "░" * (30 - bar_v) + f" {vl[-1]:.4f}")
    print()


# ══════════════════════════════════════════════
#  7. EXAMPLE USAGE
# ══════════════════════════════════════════════

if __name__ == "__main__":

    print("=" * 65)
    print("  Regularization: Dropout, L2, Early Stopping, Fit Analysis")
    print("=" * 65)

    # ── Create dataset (intentionally small → easier to overfit) ──
    torch.manual_seed(42)
    n_features = 8
    X = torch.randn(300, n_features)
    w_true = torch.randn(n_features)
    noise = torch.randn(300) * 0.3
    y = (torch.sigmoid(X @ w_true + noise) > 0.5).float().unsqueeze(1)

    # Split: 100 train (small!) / 200 val
    X_train, X_val = X[:100], X[100:]
    y_train, y_val = y[:100], y[100:]

    print(f"\n  Dataset: {X_train.shape[0]} train / {X_val.shape[0]} val / {n_features} features\n")

    # ──────────────────────────────────────────
    # Demo 1: Dropout train vs eval mode
    # ──────────────────────────────────────────
    print("═" * 65)
    print("  DEMO 1: Dropout — Train vs Inference Mode")
    print("═" * 65 + "\n")

    drop = DropoutScratch(p=0.5)
    x_demo = torch.ones(1, 8)

    drop.train()
    out_train = drop(x_demo)
    print(f"  Input:     {x_demo.squeeze().tolist()[:4]}...")
    print(f"  Train out: {out_train.squeeze().tolist()[:4]}...  (random zeros, survivors scaled ×2)")

    drop.eval()
    out_eval = drop(x_demo)
    print(f"  Eval out:  {out_eval.squeeze().tolist()[:4]}...  (unchanged — all neurons active)")

    # Verify expected value is preserved
    drop.train()
    avg = torch.stack([drop(x_demo) for _ in range(1000)]).mean(dim=0)
    print(f"  Avg over 1000 train passes: {avg.mean().item():.3f}  (≈1.0 = expected value preserved)")

    # ──────────────────────────────────────────
    # Demo 2: Overfitting vs Underfitting
    # ──────────────────────────────────────────
    print(f"\n{'═' * 65}")
    print("  DEMO 2: Overfitting vs Underfitting from Loss Curves")
    print("═" * 65)

    # A) OVERFITTING: big model, no regularization
    print(f"\n── A) Overfitting (128-128-64, no regularization) ──\n")
    torch.manual_seed(42)
    model_overfit = OverfitNet(n_features)
    hist_overfit = train_model(model_overfit, X_train, y_train, X_val, y_val,
                               epochs=200, lr=0.005, label="overfit")
    print_curves(hist_overfit, "Overfit")

    # B) UNDERFITTING: tiny model
    print(f"── B) Underfitting (2 hidden neurons) ──\n")
    torch.manual_seed(42)
    model_underfit = UnderfitNet(n_features)
    hist_underfit = train_model(model_underfit, X_train, y_train, X_val, y_val,
                                epochs=200, lr=0.005, label="underfit")
    print_curves(hist_underfit, "Underfit")

    # ──────────────────────────────────────────
    # Demo 3: Fixes — Dropout + L2 + Early Stopping
    # ──────────────────────────────────────────
    print(f"{'═' * 65}")
    print("  DEMO 3: Regularization Fixes")
    print("═" * 65)

    # C) Fix 1: Dropout
    print(f"\n── C) Fix 1: Dropout (p=0.3) ──\n")
    torch.manual_seed(42)
    model_drop = RegularizedNet(n_features, dropout=0.3)
    hist_drop = train_model(model_drop, X_train, y_train, X_val, y_val,
                            epochs=200, lr=0.005)
    print_curves(hist_drop, "Dropout")

    # D) Fix 2: L2 Regularization (weight decay)
    print(f"── D) Fix 2: L2 Weight Decay (λ=0.01) ──\n")
    torch.manual_seed(42)
    model_l2 = OverfitNet(n_features)
    hist_l2 = train_model(model_l2, X_train, y_train, X_val, y_val,
                          epochs=200, lr=0.005, weight_decay=0.01)
    print_curves(hist_l2, "L2")

    # E) Fix 3: Early Stopping
    print(f"── E) Fix 3: Early Stopping (patience=15) ──\n")
    torch.manual_seed(42)
    model_es = OverfitNet(n_features)
    es = EarlyStopping(patience=15, min_delta=1e-4)
    hist_es = train_model(model_es, X_train, y_train, X_val, y_val,
                          epochs=200, lr=0.005, early_stopping=es)
    print_curves(hist_es, "Early Stop")

    # F) All three combined
    print(f"── F) All Combined: Dropout + L2 + Early Stopping ──\n")
    torch.manual_seed(42)
    model_all = RegularizedNet(n_features, dropout=0.3)
    es_all = EarlyStopping(patience=15, min_delta=1e-4)
    hist_all = train_model(model_all, X_train, y_train, X_val, y_val,
                           epochs=200, lr=0.005, weight_decay=0.01,
                           early_stopping=es_all)
    print_curves(hist_all, "All Combined")

    # ──────────────────────────────────────────
    # Demo 4: Final comparison table
    # ──────────────────────────────────────────
    print(f"{'═' * 65}")
    print("  FINAL COMPARISON")
    print("═" * 65 + "\n")

    configs = [
        ("Overfit (no reg)",   hist_overfit),
        ("Underfit (tiny)",    hist_underfit),
        ("+ Dropout",          hist_drop),
        ("+ L2 Decay",         hist_l2),
        ("+ Early Stopping",   hist_es),
        ("+ ALL Combined",     hist_all),
    ]

    print(f"  {'Model':<22} {'Train':>7} {'Val':>7} {'Gap':>7} {'Epochs':>7} {'Diagnosis'}")
    print(f"  {'─'*22} {'─'*7} {'─'*7} {'─'*7} {'─'*7} {'─'*15}")

    for name, h in configs:
        tl = h["train_loss"][-1]
        vl = h["val_loss"][-1]
        gap = vl - tl
        ep = h["stopped_at"]

        if gap > 0.15:
            diag = "OVERFIT"
        elif tl > 0.5:
            diag = "UNDERFIT"
        else:
            diag = "GOOD ✓"

        print(f"  {name:<22} {tl:>7.3f} {vl:>7.3f} {gap:>+7.3f} {ep:>7} {diag}")

    # ── Summary ──
    print(f"""
══════════════════════════════════════════════════════════════
  REGULARIZATION CHEAT SHEET
══════════════════════════════════════════════════════════════

  OVERFITTING          │  UNDERFITTING
  train ↓↓  val ↑↑     │  train ↑  val ↑  (both high)
  gap widens           │  gap is small but losses are bad
  ─────────────────────┼──────────────────────────────────
  Fix: Dropout         │  Fix: bigger model
  Fix: L2 / weight     │  Fix: more features
       decay           │  Fix: train longer
  Fix: early stopping  │  Fix: less regularization
  Fix: more data       │  Fix: lower dropout
  Fix: simpler model   │

  DROPOUT:     Randomly drops neurons during training.
               Inverted scaling keeps expected value stable.
               .train() = random drops  │  .eval() = full network

  L2 DECAY:    Adds λ·Σw² to loss → penalizes large weights
               Equivalent to shrinking weights each step.

  EARLY STOP:  Monitor val loss, stop when it stops improving.
               Restore best checkpoint → get the best model.
══════════════════════════════════════════════════════════════
""")

  Regularization: Dropout, L2, Early Stopping, Fit Analysis

  Dataset: 100 train / 200 val / 8 features

═════════════════════════════════════════════════════════════════
  DEMO 1: Dropout — Train vs Inference Mode
═════════════════════════════════════════════════════════════════

  Input:     [1.0, 1.0, 1.0, 1.0]...
  Train out: [0.0, 2.0, 2.0, 0.0]...  (random zeros, survivors scaled ×2)
  Eval out:  [1.0, 1.0, 1.0, 1.0]...  (unchanged — all neurons active)
  Avg over 1000 train passes: 1.001  (≈1.0 = expected value preserved)

═════════════════════════════════════════════════════════════════
  DEMO 2: Overfitting vs Underfitting from Loss Curves
═════════════════════════════════════════════════════════════════

── A) Overfitting (128-128-64, no regularization) ──

  Epoch      Train Loss     Val Loss        Gap Signal
  ──────── ──────────── ──────────── ────────── ──────────────────
  1              0.7076       0.6601    -0.0475 ⚠ UNDERFITTING
  21             0.0000       0.8928